# 04 — Held-Out Test Evaluation, Literature Comparison & Error Analysis

**Project:** Obstetric Ultrasound Segmentation and Fetal Head Circumference (HC) Measurement (SBES426 — Team 2)

## Objectives
1. Run morphological cleanup, least-squares ellipse fitting (`cv2.fitEllipse`), and physical HC calculation (Ramanujan perimeter in `mm`) on all **153 unseen held-out test scans** (`117` unseen exams) at native resolution.
2. Report **pixel-level classification metrics**: Confusion Matrix, Sensitivity (Recall), Specificity, Precision, F1-Score (DSC), Accuracy, and AUC-ROC across all 66.08M test pixels.
3. Report **biometric metrics**: Dice Similarity Coefficient (DSC %), Hausdorff Distance (HD & HD95 in `mm`), Absolute Difference (AD in `mm`), and Mean Signed Difference (MD in `mm`) overall and stratified by trimester.
4. Benchmark both input arms (`raw_norm` vs. `despeckled`) against **van den Heuvel et al. (2018)** and **Sobhaninia et al. (2019)** and conduct quantitative error analysis.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import Image, display

from src.config import FIGURES_DIR, REPORTS_DIR
from src.postprocessing.run_postprocessing import run_all as run_biometrics
from src.evaluation.run_evaluation import run_all as run_pixel_eval

## 1. Postprocessing (Ellipse Fitting + Ramanujan HC) & Biometric Metrics
Evaluates both trained Baseline U-Net arms (`raw_norm` and `despeckled`) on the held-out Test split (`n = 153`), upsampling predicted probability maps bilinearly back to each scan's native resolution before morphological cleanup, `cv2.fitEllipse`, and Ramanujan HC calculation.

In [ ]:
pred_dfs, bio_summary_df, trimester_df = run_biometrics(split="test")
display(bio_summary_df)
display(trimester_df)

## 2. Qualitative Ellipse Overlays & Bland-Altman Clinical Agreement
Visualizes 1st, 2nd, and 3rd trimester scans plus the highest-error case (ground truth vs. cleaned U-Net mask vs. least-squares fitted ellipse), alongside HC agreement scatter and Bland-Altman plots.

In [ ]:
display(Image(filename=str(FIGURES_DIR / "postprocessing_ellipse_overlays.png")))
display(Image(filename=str(FIGURES_DIR / "biometric_bland_altman_and_scatter.png")))

## 3. Pixel-Level Classification Metrics (Confusion Matrix, Sensitivity, Specificity, F1, AUC-ROC)
Evaluates pixel classification across all 66,083,122 native-resolution test pixels and plots side-by-side Confusion Matrices and ROC curves.

In [ ]:
pixel_df, lit_df, err_df = run_pixel_eval(split="test")
display(pixel_df)
display(Image(filename=str(FIGURES_DIR / "confusion_matrix.png")))
display(Image(filename=str(FIGURES_DIR / "roc_curves.png")))

## 4. Literature Comparison & Error Analysis
Compares both baseline arms against **van den Heuvel et al. (2018)** and **Sobhaninia et al. (2019)** and inspects the top-8 largest-error test cases to guide Phase 3 improvements.

In [ ]:
display(lit_df)
display(Image(filename=str(FIGURES_DIR / "literature_comparison.png")))

print("Top-8 Largest-Error Test Cases:")
display(err_df)